In [4]:
!wget -O ans.txt https://raw.githubusercontent.com/3b1b/videos/master/_2022/wordle/data/possible_words.txt

--2026-11-22 14:03:06--  https://raw.githubusercontent.com/3b1b/videos/master/_2022/wordle/data/possible_words.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13854 (14K) [text/plain]
Saving to: ‘ans.txt’

ans.txt             100%[===================>]  13.53K  --.-KB/s    in 0.001s  

2026-11-22 14:03:07 (16.9 MB/s) - ‘ans.txt’ saved [13854/13854]



In [2]:
!ls -l ans.txt
!head -5 ans.txt
!wc -l ans.txt

-rw-r--r-- 1 root root 13854 Nov 22 13:36 ans.txt
aback
abase
abate
abbey
abbot
2309 ans.txt


In [3]:
import urllib.request, os
url = "https://raw.githubusercontent.com/3b1b/videos/master/_2022/wordle/data/possible_words.txt"
if os.path.exists("ans.txt"):
    os.remove("ans.txt")
urllib.request.urlretrieve(url, "ans.txt")
words = [w.strip() for w in open("ans.txt") if len(w.strip()) == 5]
print(len(words), words[:5])

2309 ['aback', 'abase', 'abate', 'abbey', 'abbot']


In [ ]:
import numpy as np, math, random, sys
W=[w.strip().lower() for w in open('ans.txt') if len(w.strip())==5]
W=sorted(set(W)); M=len(W); print("M =",M)
def pat(g,t):
    r=[0]*5; cnt={}
    for i in range(5):
        if g[i]==t[i]: r[i]=2
        else: cnt[t[i]]=cnt.get(t[i],0)+1
    for i in range(5):
        if r[i]==0 and cnt.get(g[i],0)>0: r[i]=1; cnt[g[i]]-=1
    return sum(r[i]*3**i for i in range(5))
P=np.zeros((M,M),dtype=np.uint8)
for i,g in enumerate(W):
    for j,t in enumerate(W): P[i,j]=pat(g,t)
np.save('P.npy',P)
def entropy_of(g,cand):
    c=np.bincount(P[g,cand],minlength=243); c=c[c>0]; p=c/c.sum()
    return -(p*np.log2(p)).sum()
allidx=np.arange(M)
H=np.array([entropy_of(g,allidx) for g in range(M)])
order=np.argsort(-H)
print("H(X)=log2 M =",math.log2(M))
print("Top 5 first words:",[(W[i],round(H[i],3)) for i in order[:5]])
first=order[0]
def play_shannon(t,first=first):
    cand=allidx.copy(); g=first; n=0
    while True:
        n+=1
        if g==t: return n
        cand=cand[P[g,cand]==P[g,t]]
        if len(cand)==1: g=cand[0]; continue
        cs=set(cand.tolist())
        best=None;bh=-1
        for k in (cand if len(cand)>60 else allidx):
            h=entropy_of(k,cand)+(1e-9 if k in cs else 0)
            if h>bh: bh=h;best=k
        g=best
        if n>=15: return n
def play_random(t,rng,fixed=None):
    cand=allidx.copy(); n=0; g=fixed if fixed is not None else rng.choice(cand)
    while True:
        n+=1
        if g==t: return n
        cand=cand[P[g,cand]==P[g,t]]
        g=rng.choice(cand)
        if n>=15: return n
rng=np.random.default_rng(1)
def stats(res):
    res=np.array(res); return res.mean(), (res<=6).mean()*100, res.max(), np.bincount(res)
rs=[play_shannon(t) for t in range(M)]
print("SHANNON",stats(rs))
rr=[play_random(t,rng) for t in range(M)]
print("RANDOM",stats(rr))
fx=W.index('crane') if 'crane' in W else None
if fx is not None:
    rf=[play_random(t,rng,fx) for t in range(M)]
    print("FIXED crane + random",stats(rf))
# worked example
tgt=W.index('shard') if 'shard' in W else 0
cand=allidx.copy(); g=first
print("TARGET",W[tgt])
for step in range(1,4):
    pt=P[g,tgt]; Hc=-sum((c/len(cand))*math.log2(c/len(cand)) for c in np.bincount(P[g,cand],minlength=243) if c)
    new=cand[P[g,cand]==pt]
    sym=''.join('GYB'[ [2,1,0].index(v) ] if False else '⬛🟨🟩'[v] for v in [(pt//3**i)%3 for i in range(5)])
    print(step,W[g],sym,"before",len(cand),"after",len(new),"H(X)=%.3f"%math.log2(len(cand)),"E[I]=%.3f"%Hc,"I_real=%.3f"%(math.log2(len(cand))-math.log2(len(new))),"H(X|Y)exp=%.3f"%(math.log2(len(cand))-Hc))
    cand=new
    if len(cand)==1 or W[g]==W[tgt]: break
    cs=set(cand.tolist()); best=None;bh=-1
    for k in (cand if len(cand)>60 else allidx):
        h=entropy_of(k,cand)+(1e-9 if k in cs else 0)
        if h>bh: bh=h;best=k
    g=best

M = 2309
H(X)=log2 M = 11.173052457774116
Top 5 first words: [('raise', np.float64(5.878)), ('slate', np.float64(5.856)), ('crate', np.float64(5.835)), ('irate', np.float64(5.833)), ('trace', np.float64(5.83))]
SHANNON (np.float64(3.5058466868774363), np.float64(100.0), np.int64(6), array([   0,    1,   67, 1100, 1047,   92,    2]))
RANDOM (np.float64(4.074924209614552), np.float64(98.13772195755739), np.int64(9), array([  0,   0, 101, 564, 932, 540, 129,  33,   8,   2]))
FIXED crane + random (np.float64(3.8531831961888265), np.float64(98.44088349935038), np.int64(8), array([  0,   1, 149, 742, 890, 392,  99,  30,   6]))
TARGET shard
1 raise 🟨🟨⬛🟨⬛ before 2309 after 21 H(X)=11.173 E[I]=5.878 I_real=6.781 H(X|Y)exp=5.295
2 splat 🟩⬛⬛🟨⬛ before 21 after 5 H(X)=4.392 E[I]=3.423 I_real=2.070 H(X|Y)exp=0.970
3 duchy 🟨⬛⬛🟨⬛ before 5 after 1 H(X)=2.322 E[I]=2.322 I_real=2.322 H(X|Y)exp=0.000
